In [143]:
import os
os.environ["HTTP_PROXY"]= "http://proxy.utwente.nl:3128"
os.environ["HTTPS_PROXY"]= "http://proxy.utwente.nl:3128"
os.environ["http_proxy"]= "http://proxy.utwente.nl:3128"
os.environ["https_proxy"]= "http://proxy.utwente.nl:3128"

MITRE augmentation tram



In [144]:
import pandas as pd
from sklearn.model_selection import train_test_split

df_train = pd.read_json("datasets/bosch_train.json")



In [145]:
from collections import Counter

label_counts = Counter(
    label
    for labels in df_train["labels"]
    for label in labels
)

label_distribution = (
    pd.DataFrame.from_dict(label_counts, orient="index", columns=["count"])
    .sort_values(by="count", ascending=False)
)
print(label_distribution.head())


        count
TA0011    407
T1566     273
T1059     206
T1486     168
T1105     117


In [146]:
import json
import pandas as pd
# 1) Load cleaned.json
with open("../../data_augmentatio_stefano/mitre/mitre_relationships.json", "r", encoding="utf-8") as f:
    cleaned = json.load(f)
print(type(cleaned), len(cleaned))  # sanity check: should be (list, N)

# 2) Build ood_data from the cleaned entries
ood_data = {
    "sentence": [],
    "labels": [],
    "doc_title": [],
}

for i, item in enumerate(cleaned):
    # Extract the text and label
    sent = item.get("relationship_description", "")
    ttp_id = item.get("technique_id")
    doc_title = item.get("source_type", "")

    # Basic sanity filtering
    if not sent or not ttp_id:
        continue

    # Append to OOD structure
    ood_data["sentence"].append(sent)
    ood_data["labels"].append([ttp_id])
    ood_data["doc_title"].append(doc_title + str(i))

# 3) Make the DataFrame
ood_data_df = pd.DataFrame(ood_data)
print(ood_data_df.shape)
ood_data_df.head()


<class 'list'> 19280
(19280, 3)


,sentence,labels,doc_title
0,Adversaries may inject malicious code into pro...,[T1055.011],attack-pattern0
1,Adversaries may abuse the Windows Task Schedul...,[T1053.005],attack-pattern1
2,Adversaries may attach filters to a network so...,[T1205.002],attack-pattern2
3,If a malicious tool is detected and quarantine...,[T1066],attack-pattern3
4,Adversaries may use utilities to compress and/...,[T1560.001],attack-pattern4


In [147]:
# Keep only OOD sentences whose labels appear in TRAM
tram_labels = set(label_counts.keys())

selected_ood_data_df = ood_data_df[
    ood_data_df["labels"].apply(lambda x: any(label in tram_labels for label in x))
].copy()

selected_ood_data_df.head(), selected_ood_data_df.shape


(                                             sentence   labels  \
 6   Adversaries may abuse Windows Management Instr...  [T1047]   
 8   Adversaries may attempt to take screen capture...  [T1113]   
 11  Adversaries may use scripts automatically exec...  [T1037]   
 12  Adversaries may attempt to position themselves...  [T1557]   
 13  Adversaries may attempt to identify the primar...  [T1033]   
 
            doc_title  
 6    attack-pattern6  
 8    attack-pattern8  
 11  attack-pattern11  
 12  attack-pattern12  
 13  attack-pattern13  ,
 (6584, 3))

In [148]:
# Group by sentence, merge label lists and de-duplicate within each sentence
merged_ood_data_df = (
    selected_ood_data_df
    .groupby("sentence", as_index=False)
    .agg({
        "labels": lambda x: list({l for sublist in x for l in sublist}),
        "doc_title": lambda x: list(x.unique())
    })
)


merged_ood_data_df.columns = ["sentence", "labels", "doc_title"]
merged_ood_data_df.head(), merged_ood_data_df.shape


(                                            sentence   labels  \
 0  1) New or updated software is delivered/instal...  [T1195]   
 1  3PARA RAT has a command to retrieve metadata f...  [T1083]   
 2  4H RAT has the capability to obtain a listing ...  [T1057]   
 3  4H RAT has the capability to obtain file and d...  [T1083]   
 4  4H RAT sends an OS version identifier in its b...  [T1082]   
 
                           doc_title  
 0  [x-mitre-detection-strategy7647]  
 1                     [malware9740]  
 2                     [malware8976]  
 3                     [malware3594]  
 4                     [malware3633]  ,
 (6535, 3))

In [149]:
merged_ood_data_df.to_json("datasets/bosch_train_augmented_mitre.json")


In [150]:
label_counts = Counter(
    label
    for labels in df_train["labels"]
    for label in labels
)
print(label_counts)
label_distribution = (
    pd.DataFrame.from_dict(label_counts, orient="index", columns=["count"])
    .sort_values(by="count", ascending=False)
)
print("TRAM\n",label_distribution.tail())

label_counts = Counter(
    label
    for labels in merged_ood_data_df["labels"]
    for label in labels
)
print(label_counts)
label_distribution = (
    pd.DataFrame.from_dict(label_counts, orient="index", columns=["count"])
    .sort_values(by="count", ascending=False)
)
print("\nTRAM_aug_MITRE\n",label_distribution.tail())

Counter({'TA0011': 407, 'T1566': 273, 'T1059': 206, 'T1486': 168, 'T1105': 117, 'S0154': 84, 'S0266': 76, 'T1071': 75, 'T1027': 65, 'T1140': 64, 'S0534': 61, 'G0044': 55, 'T1041': 46, 'G0059': 46, 'T1190': 44, 'T1573': 44, 'S0367': 40, 'T1036': 40, 'G0007': 39, 'TA0006': 39, 'T1056': 39, 'TA0010': 38, 'T1204': 36, 'S0648': 34, 'T1005': 33, 'TA0003': 30, 'TA0002': 30, 'T1496': 30, 'G0003': 30, 'T1055': 28, 'T1555': 28, 'G0065': 27, 'T1499': 24, 'T1082': 24, 'T1195': 22, 'G0080': 21, 'T1497': 20, 'T1547': 19, 'T1189': 19, 'S0198': 19, 'S0262': 19, 'T1589': 18, 'S0496': 17, 'TA0009': 17, 'T1110': 17, 'T1113': 16, 'T1562': 15, 'TA0005': 14, 'T1587': 14, 'S0446': 13, 'T1203': 12, 'TA0001': 12, 'T1078': 12, 'T1083': 12, 'T1125': 12, 'T1053': 12, 'S0332': 12, 'G0096': 12, 'S0650': 12, 'T1218': 11, 'T1543': 11, 'G0032': 10, 'T1014': 10, 'S0226': 10, 'S0554': 9, 'T1571': 9, 'T1112': 9, 'G0046': 9, 'T1033': 9, 'T1132': 9, 'S0183': 8, 'TA0007': 8, 'T1021': 8, 'T1057': 8, 'T1090': 8, 'T1490': 8, '

MITRE augmentation BOSCH


In [151]:
import pandas as pd
from sklearn.model_selection import train_test_split

df_train = pd.read_json("datasets/tram_train.json")



In [152]:
from collections import Counter

label_counts = Counter(
    label
    for labels in df_train["labels"]
    for label in labels
)

label_distribution = (
    pd.DataFrame.from_dict(label_counts, orient="index", columns=["count"])
    .sort_values(by="count", ascending=False)
)
print(label_distribution.head())


           count
T1027        557
T1140        386
T1059.003    293
T1055        237
T1105        201


In [153]:
import json
import pandas as pd
# 1) Load cleaned.json
with open("../../data_augmentatio_stefano/mitre/mitre_relationships.json", "r", encoding="utf-8") as f:
    cleaned = json.load(f)
print(type(cleaned), len(cleaned))  # sanity check: should be (list, N)

# 2) Build ood_data from the cleaned entries
ood_data = {
    "sentence": [],
    "labels": [],
    "doc_title": [],
}

for i, item in enumerate(cleaned):
    # Extract the text and label
    sent = item.get("relationship_description", "")
    ttp_id = item.get("technique_id")
    doc_title = item.get("source_type", "") + str(i)

    # Basic sanity filtering
    if not sent or not ttp_id:
        continue

    # Append to OOD structure
    ood_data["sentence"].append(sent)
    ood_data["labels"].append([ttp_id])
    ood_data["doc_title"].append(doc_title)

# 3) Make the DataFrame
ood_data_df = pd.DataFrame(ood_data)
print(ood_data_df.shape)
ood_data_df.head()


<class 'list'> 19280
(19280, 3)


,sentence,labels,doc_title
0,Adversaries may inject malicious code into pro...,[T1055.011],attack-pattern0
1,Adversaries may abuse the Windows Task Schedul...,[T1053.005],attack-pattern1
2,Adversaries may attach filters to a network so...,[T1205.002],attack-pattern2
3,If a malicious tool is detected and quarantine...,[T1066],attack-pattern3
4,Adversaries may use utilities to compress and/...,[T1560.001],attack-pattern4


In [154]:
# Keep only OOD sentences whose labels appear in TRAM
tram_labels = set(label_counts.keys())

selected_ood_data_df = ood_data_df[
    ood_data_df["labels"].apply(lambda x: any(label in tram_labels for label in x))
].copy()

selected_ood_data_df.head(), selected_ood_data_df.shape


(                                             sentence       labels  \
 1   Adversaries may abuse the Windows Task Schedul...  [T1053.005]   
 6   Adversaries may abuse Windows Management Instr...      [T1047]   
 8   Adversaries may attempt to take screen capture...      [T1113]   
 13  Adversaries may attempt to identify the primar...      [T1033]   
 15  Adversaries may abuse rundll32.exe to proxy ex...  [T1218.011]   
 
            doc_title  
 1    attack-pattern1  
 6    attack-pattern6  
 8    attack-pattern8  
 13  attack-pattern13  
 15  attack-pattern15  ,
 (7947, 3))

In [155]:
# Group by sentence, merge label lists and de-duplicate within each sentence
merged_ood_data_df = (
    selected_ood_data_df
    .groupby("sentence", as_index=False)
    .agg({
        "labels": lambda x: list({l for sublist in x for l in sublist}),
        "doc_title": lambda x: list(x.unique())
    })
)


merged_ood_data_df.columns = ["sentence", "labels", "doc_title"]
merged_ood_data_df.head(), merged_ood_data_df.shape


(                                            sentence       labels  \
 0  3PARA RAT command and control commands are enc...  [T1573.001]   
 1  3PARA RAT has a command to retrieve metadata f...      [T1083]   
 2       3PARA RAT uses HTTP for command and control.  [T1071.001]   
 3  4H RAT has the capability to create a remote s...  [T1059.003]   
 4  4H RAT has the capability to obtain a listing ...      [T1057]   
 
        doc_title  
 0  [malware1823]  
 1  [malware9740]  
 2  [malware4039]  
 3  [malware5670]  
 4  [malware8976]  ,
 (7929, 3))

In [156]:
merged_ood_data_df.to_json("datasets/tram_train_augmented_mitre.json")


Check dataset:

In [135]:
from tqdm import tqdm

def validate_bosch_json_allow_empty(labels_dict):
    """
    Validates BOSCH-style labels dict.
    Empty label lists are allowed.
    """

    print("🔍 Validating labels (empty lists allowed)...")
    bad = []

    for idx in tqdm(labels_dict.keys(), desc="Checking labels"):
        labels = labels_dict[idx]

        try:
            # ❌ None is invalid
            if labels is None:
                raise ValueError("labels is None")

            # ❌ string instead of list
            if isinstance(labels, str):
                raise ValueError(f"labels is str: {labels}")

            # ❌ not list / tuple
            if not isinstance(labels, (list, tuple)):
                raise ValueError(f"labels type is {type(labels)}")

            # ✅ empty list is OK → skip further checks
            if len(labels) == 0:
                continue

            # ❌ nested lists or non-string labels
            for l in labels:
                if not isinstance(l, str):
                    raise ValueError(f"label {l} has type {type(l)}")

        except Exception as e:
            bad.append((idx, labels, str(e)))

    print(f"\n❌ Found {len(bad)} problematic samples\n")

    for idx, labels, err in bad[:20]:
        print(f"[idx={idx}] labels={labels} → {err}")

    if len(bad) > 20:
        print(f"... {len(bad) - 20} more")

    return bad


In [136]:
import json

with open("datasets/bosch_train_augmented_mitre.json", "r", encoding="utf-8") as f:
    data = json.load(f)

bad_samples = validate_bosch_json_allow_empty(data["labels"])



🔍 Validating labels (empty lists allowed)...


Checking labels: 100%|██████████| 7929/7929 [00:00<00:00, 678485.32it/s]


❌ Found 0 problematic samples

